# Notebook 2: Robots Validation, Relevance-Guided Website Crawl, and Translation

This notebook replaces the previous one-layer flow:

`homepage → extract links → select top 1 → fetch → stop`

with a bounded graph traversal:

`homepage → discover internal links → score links → queue relevant unseen pages → fetch them → repeat`

The crawler uses:

- same-domain URL validation
- `robots.txt` checks for each crawled path
- a BFS queue ordered by depth
- cosine similarity to filter and rank links within each depth
- duplicate protection through `visited` and `queued` sets
- strict page, depth, and branching limits
- actual page-content scoring to select one final candidate per company

In [47]:
from __future__ import annotations

import asyncio
import csv
import heapq
import re
from collections import deque
from pathlib import Path
from typing import Any
from urllib.parse import urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import httpx
import numpy as np
import pandas as pd
import pycountry
import torch
from bs4 import BeautifulSoup
from langdetect import DetectorFactory, detect
from sentence_transformers import SentenceTransformer, util

## 1. Configuration

In [ ]:
INPUT_COMPANY_CSV = Path("../data/processed/company_scrape_ready.csv")
ROBOTS_OUTPUT_CSV = Path("../data/processed/robot_check.csv")
HOMEPAGE_OUTPUT_CSV = Path("../data/processed/homepage_fetch.csv")
CRAWL_AUDIT_OUTPUT_CSV = Path("../data/processed/crawled_pages.csv")
PRODUCTION_FINAL_CSV = Path("../data/processed/production_final.csv")
TRANSLATION_OUTPUT_CSV = Path("../data/processed/production_500_sample.csv")
TRANSLATION_CACHE_CSV = Path("../data/processed/translation_cache.csv")

USER_AGENT = "CI-ResearchBot/1.0"
REQUEST_TIMEOUT_SECONDS = 15
MAX_CONCURRENT_REQUESTS = 10

# Crawl limits prevent runaway websites, pagination traps, and excessive requests.
MAX_CRAWL_DEPTH = 3
MAX_PAGES_PER_COMPANY = 12
MAX_CHILDREN_PER_PAGE = 2

MIN_CONTENT_SIMILARITY = 0.44
MIN_SEMANTIC_MARGIN = 0.12
MIN_LINK_SIMILARITY = 0.38

MIN_PAGE_TEXT_CHARACTERS = 300
MIN_PAGE_TEXT_WORDS = 40

MAX_PAGE_TEXT_CHARACTERS = 10_000
CONTENT_SCORING_CHARACTERS = 10_000

# One variable controls how many companies are processed.
# Use a positive integer such as 5, 50, or 500.
# Use "full" to process every company.
RUN_SIZE: int | str = 500

# Set a flag to True only when that stage must be run again.
RERUN_ROBOTS = False
RERUN_HOMEPAGES = False
RERUN_CRAWL = True
RERUN_TRANSLATION = False

RANDOM_SEED = 42

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

TARGET_LINK_THEMES = [
    "our core values",
    "company core values",
    "corporate values",
    "organisational values",
    "organizational values",
    "company principles",
    "guiding principles",
    "core principles",
    "our beliefs",
    "company beliefs",
    "what we believe",
    "our purpose",
    "company purpose",
    "our mission",
    "company mission",
    "our vision",
    "company vision",
    "mission vision and values",
    "purpose mission vision and values",
    "our culture",
    "company culture",
    "workplace culture",
    "organisational culture",
    "organizational culture",
    "shared values",
    "values and behaviours",
    "values and behaviors",
    "our commitments",
    "company commitments",
    "ethical principles",
    "business ethics",
    "code of values",
    "principles that guide us",
    "what guides our decisions",
    "how we work",
    "how we behave",
    "who we are and what we stand for",
    "what we stand for",
    "our identity and values",
    "our philosophy",
    "company philosophy",
    "our ethos",
    "corporate ethos",
    "our DNA",
    "company DNA",
    "our way",
    "the way we work",
    "our standards",
    "company standards",
    "our foundations",
    "our values and culture",
    "our mission and values",
    "our vision and values",
    "our purpose and values",
    "our principles and values",
]
NEGATIVE_CONTENT_THEMES = [
    (
        "A page advertising leadership training, management development, "
        "coaching, workshops, academies, courses, or professional education services."
    ),
    (
        "A page describing a commercial training programme, academy, consulting "
        "service, learning product, or customer offering."
    ),
    (
        "A page explaining how a service helps customers develop leadership, "
        "culture, vision, behaviours, purpose, or organisational values."
    ),
    (
        "The page mainly discusses corporate governance, board structure, "
        "executive oversight, committees, compliance, or shareholder responsibilities."
    ),
    (
        "The page mainly contains investor relations information, annual reports, "
        "financial performance, stock information, or corporate disclosures."
    ),
    (
        "The page mainly provides contact details, office locations, enquiry forms, "
        "phone numbers, email addresses, or customer support information."
    ),
    (
        "The page mainly lists jobs, vacancies, recruitment information, employee "
        "benefits, or career opportunities."
    ),
    (
        "The page mainly describes products, services, industries, capabilities, "
        "customer solutions, or commercial offerings."
    ),
    (
        "The page mainly contains news articles, press releases, media updates, "
        "events, announcements, or blog posts."
    ),
]

TARGET_CONTENT_THEMES = [
    (
        "The company explicitly explains its core values, guiding principles, "
        "beliefs, or standards that influence employee behaviour and business decisions."
    ),
    (
        "The page describes the organisation's mission, vision, and purpose, "
        "including what the company aims to achieve and why it exists."
    ),
    (
        "The company presents named values such as integrity, respect, innovation, "
        "collaboration, accountability, trust, excellence, inclusion, or sustainability."
    ),
    (
        "The page explains the behaviours expected from employees and how the "
        "organisation's values guide everyday work."
    ),
    (
        "The company describes its organisational culture, shared values, workplace "
        "beliefs, and the principles that shape how people work together."
    ),
    (
        "The page contains a structured list of company values, each supported by "
        "an explanation, definition, example, or behavioural statement."
    ),
    (
        "The company explains what it stands for, what it believes in, and the "
        "principles used to guide choices and relationships with stakeholders."
    ),
    (
        "The organisation defines its ethical principles, commitments, or standards "
        "of conduct as part of its identity and culture."
    ),
    (
        "The page explains the company's philosophy, ethos, identity, or DNA in "
        "terms of values, beliefs, purpose, and expected behaviour."
    ),
    (
        "The company describes how its mission, purpose, vision, culture, and values "
        "connect to its long-term direction and decision-making."
    ),
]

EXCLUDED_PATH_PATTERNS = [
    r"/contact(?:/|$)",
    r"/privacy(?:/|$)",
    r"/terms(?:/|$)",
    r"/terms-of-use(?:/|$)",
    r"/termos-de-uso(?:/|$)",
    r"/legal(?:/|$)",
    r"/login(?:/|$)",
    r"/signin(?:/|$)",
    r"/search(?:/|$)",
    r"/cart(?:/|$)",
    r"/checkout(?:/|$)",
    r"/author(?:/|$)",
    r"/tag(?:/|$)",
    r"/category(?:/|$)",
    r"/news(?:/|$)",
    r"/blog(?:/|$)",
    r"/press(?:/|$)",
    r"/media(?:/|$)",
    r"/jobs?(?:/|$)",
    r"/careers?(?:/|$)",
    r"/vacanc(?:y|ies)(?:/|$)",
    r"/recruitment(?:/|$)",
    r"/board-of-directors(?:/|$)",
    r"/leadership-team(?:/|$)",
    r"/management-team(?:/|$)",
    r"/corporate-governance(?:/|$)",
    r"/investor-relations(?:/|$)",
    r"/awards?(?:/|$)",
    r"/milestones?(?:/|$)",
    r"/clients?(?:/|$)",
    r"/customers?(?:/|$)",
    r"/partners?(?:/|$)",
    r"/brands?(?:/|$)",
    r"/franchises?(?:/|$)",
    r"/groupcompanies(?:/|$)",
    r"/group-companies(?:/|$)",
    r"/team(?:/|$)",
    r"/our-team(?:/|$)",
    r"/people(?:/|$)",
    r"/certifications?(?:/|$)",
    r"/accreditations?(?:/|$)",
    r"/quality-assurance(?:/|$)",
    r"/quality(?:/|$)",
    r"/products?(?:/|$)",
    r"/services?(?:/|$)",
    r"/solutions?(?:/|$)",
    r"/industries?(?:/|$)",
    r"/case-studies?(?:/|$)",
    r"/resources?(?:/|$)",
    r"/events?(?:/|$)",
    r"/mental-health(?:/|$)",
]

EXCLUDED_EXTENSIONS = {
    ".pdf", ".jpg", ".jpeg", ".png", ".gif", ".svg",
    ".zip", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
}

ANCHOR_TEXT_NOISE = {
    "", "skip to content", "skip to main content", "click here",
    "read more", "learn more", "more", "menu", "close",
    "next", "previous", "back", "top",
}

for path in [
    ROBOTS_OUTPUT_CSV,
    HOMEPAGE_OUTPUT_CSV,
    CRAWL_AUDIT_OUTPUT_CSV,
    PRODUCTION_FINAL_CSV,
    TRANSLATION_OUTPUT_CSV,
    TRANSLATION_CACHE_CSV,
]:
    path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load and prepare company input

In [49]:
df_companies_raw = pd.read_csv(INPUT_COMPANY_CSV)

required_columns = [
    "company_id",
    "name",
    "country",
    "size_bucket",
    "domain",
    "final_url",
]

missing_columns = [
    column for column in required_columns
    if column not in df_companies_raw.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_companies = (
    df_companies_raw[required_columns]
    .dropna(subset=["company_id", "final_url"])
    .drop_duplicates(subset=["company_id"], keep="first")
    .reset_index(drop=True)
)

if isinstance(RUN_SIZE, int):
    if RUN_SIZE <= 0:
        raise ValueError("RUN_SIZE must be greater than 0.")

    df_companies = (
        df_companies.sample(
            n=min(RUN_SIZE, len(df_companies)),
            random_state=RANDOM_SEED,
        )
        .sort_values("company_id")
        .reset_index(drop=True)
    )

elif isinstance(RUN_SIZE, str) and RUN_SIZE.lower() == "full":
    df_companies = df_companies.reset_index(drop=True)

else:
    raise ValueError("RUN_SIZE must be a positive integer or 'full'.")

print("Companies selected:", len(df_companies))
df_companies.head(3)

Companies selected: 500


,company_id,name,country,size_bucket,domain,final_url
0,C0001,australian beef sustainability framework,australia,enterprise,sustainableaustralianbeef.com.au,https://www.sustainableaustralianbeef.com.au/
1,C0003,austal,australia,enterprise,austal.com,https://www.austal.com/
2,C0004,carzoos,australia,enterprise,carzoos.com.au,https://easyauto123.com.au/


## 3. Shared URL, HTML, and feature helpers

In [50]:
def normalise_domain(url: str) -> str:
    parsed_url = urlparse(str(url))
    domain = parsed_url.netloc.lower().strip()

    if domain.startswith("www."):
        domain = domain[4:]

    return domain


def clean_internal_url(url: str) -> str:
    parsed_url = urlparse(str(url).strip())

    cleaned_url = parsed_url._replace(
        query="",
        fragment="",
    )

    return urlunparse(cleaned_url).rstrip("/")


def get_url_path(url: str) -> str:
    path = urlparse(str(url)).path
    return path if path else "/"


def is_same_homepage_domain(url: str, homepage_url: str) -> bool:
    link_domain = normalise_domain(url)
    homepage_domain = normalise_domain(homepage_url)

    return (
        link_domain == homepage_domain
        or link_domain.endswith("." + homepage_domain)
    )


def clean_anchor_text(anchor_text: str) -> str:
    cleaned = re.sub(r"\s+", " ", str(anchor_text)).strip()
    return cleaned[:300]


def clean_page_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script",
        "style",
        "noscript",
        "header",
        "footer",
        "nav",
        "form",
        "aside",
        "iframe",
        "svg",
        "button",
    ]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text[:MAX_PAGE_TEXT_CHARACTERS]


def is_excluded_url(url: str) -> bool:
    path = get_url_path(url).lower()

    if any(path.endswith(extension) for extension in EXCLUDED_EXTENSIONS):
        return True

    return any(
        re.search(pattern, path)
        for pattern in EXCLUDED_PATH_PATTERNS
    )


def improved_extract_path_features(url_path: str) -> str:
    if pd.isna(url_path):
        return ""

    path = str(url_path).lower().strip()
    path = re.sub(r"\.[a-zA-Z0-9]+$", "", path)

    noise_words = [
        "index",
        "home",
        "page",
        "wp-content",
        "uploads",
        "assets",
        "attachment",
    ]

    for noise_word in noise_words:
        path = path.replace(noise_word, " ")

    words = re.sub(r"[/_-]+", " ", path)

    abbreviations = {
    r"\bco\b": "company",
    r"\babt\b": "about",
    r"\bprofile\b": "company profile",
    r"\bvis\b": "vision",
    r"\bval\b": "values",
}

    for pattern, replacement in abbreviations.items():
        words = re.sub(pattern, replacement, words)

    return " ".join(words.split())


def build_feature_text(anchor_text: str, url_path: str) -> str:
    anchor = clean_anchor_text(anchor_text)
    path_features = improved_extract_path_features(url_path)

    if anchor.lower() in ANCHOR_TEXT_NOISE:
        anchor = ""

    if anchor and path_features:
        return f"{anchor} ({path_features})"

    return anchor or path_features


def extract_internal_links_from_page(
    html: str,
    current_page_url: str,
    allowed_domain_url: str,
) -> list[dict[str, str]]:
    soup = BeautifulSoup(html, "html.parser")
    links: list[dict[str, str]] = []

    for anchor in soup.find_all("a", href=True):
        raw_link = anchor.get("href")

        if not raw_link:
            continue

        absolute_url = urljoin(current_page_url, raw_link)
        internal_url = clean_internal_url(absolute_url)

        if not internal_url.startswith(("https://", "http://")):
            continue

        if not is_same_homepage_domain(internal_url, allowed_domain_url):
            continue

        if is_excluded_url(internal_url):
            continue

        links.append({
            "raw_link": raw_link,
            "internal_url": internal_url,
            "url_path": get_url_path(internal_url),
            "anchor_text": clean_anchor_text(anchor.get_text(" ")),
        })

    # Preserve the first useful anchor while removing duplicate destination URLs.
    unique_links: list[dict[str, str]] = []
    seen_urls: set[str] = set()

    for link in links:
        if link["internal_url"] in seen_urls:
            continue

        seen_urls.add(link["internal_url"])
        unique_links.append(link)

    return unique_links

## 4. Ticket 10268: robots.txt validation

In [51]:
def build_robots_url(final_url: str) -> str:
    parsed_url = urlparse(final_url)
    return f"{parsed_url.scheme}://{parsed_url.netloc}/robots.txt"


async def fetch_robots_text(
    client: httpx.AsyncClient,
    robots_url: str,
) -> tuple[int | None, str | None, str | None]:
    try:
        response = await client.get(
            robots_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        return response.status_code, response.text, None

    except httpx.HTTPError as error:
        return None, None, f"robots_fetch_error_{type(error).__name__}"


def build_robots_parser(
    robots_url: str,
    robots_text: str | None,
) -> RobotFileParser | None:
    if not robots_text:
        return None

    parser = RobotFileParser()
    parser.set_url(robots_url)
    parser.parse(robots_text.splitlines())

    return parser


async def validate_company_robots(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    final_url = company["final_url"]
    robots_url = build_robots_url(final_url)

    async with request_semaphore:
        status_code, robots_text, fetch_error = await fetch_robots_text(
            client=client,
            robots_url=robots_url,
        )

    if fetch_error:
        robots_allowed = True
        robots_reason = fetch_error
    elif status_code is not None and status_code >= 400:
        robots_allowed = True
        robots_reason = f"robots_status_{status_code}"
    else:
        try:
            parser = build_robots_parser(robots_url, robots_text)
            robots_allowed = (
                True
                if parser is None
                else parser.can_fetch(USER_AGENT, final_url)
            )
            robots_reason = "allowed" if robots_allowed else "blocked"
        except Exception as error:
            robots_allowed = True
            robots_reason = f"robots_parse_error_{type(error).__name__}"

    return {
        "company_id": company["company_id"],
        "name": company["name"],
        "country": company["country"],
        "size_bucket": company["size_bucket"],
        "domain": company["domain"],
        "final_url": final_url,
        "robots_url": robots_url,
        "robots_status_code": status_code,
        "robots_allowed": robots_allowed,
        "robots_reason": robots_reason,
    }


async def run_robots_validation(df_input: pd.DataFrame) -> pd.DataFrame:
    request_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT},
        follow_redirects=True,
    ) as client:
        tasks = [
            validate_company_robots(
                company=row,
                client=client,
                request_semaphore=request_semaphore,
            )
            for _, row in df_input.iterrows()
        ]

        results = await asyncio.gather(*tasks)

    return pd.DataFrame(results)
if ROBOTS_OUTPUT_CSV.exists() and not RERUN_ROBOTS:
    df_robot_cached = pd.read_csv(ROBOTS_OUTPUT_CSV)

    requested_company_ids = set(df_companies["company_id"])
    cached_company_ids = set(df_robot_cached["company_id"])
    missing_company_ids = requested_company_ids - cached_company_ids

    df_robot_validated = df_robot_cached[
        df_robot_cached["company_id"].isin(requested_company_ids)
    ].copy()

    if missing_company_ids:
        df_missing_robots = df_companies[
            df_companies["company_id"].isin(missing_company_ids)
        ].copy()

        df_new_robots = await run_robots_validation(df_missing_robots)

        df_robot_validated = pd.concat(
            [df_robot_validated, df_new_robots],
            ignore_index=True,
        )

        df_robot_cached = pd.concat(
            [df_robot_cached, df_new_robots],
            ignore_index=True,
        ).drop_duplicates(
            subset=["company_id"],
            keep="last",
        )

        df_robot_cached.to_csv(
            ROBOTS_OUTPUT_CSV,
            index=False,
        )

    print("Loaded robots results from cache.")

else:
    df_robot_validated = await run_robots_validation(df_companies)
    df_robot_validated.to_csv(ROBOTS_OUTPUT_CSV, index=False)

print("Robots allowed:", int(df_robot_validated["robots_allowed"].sum()))
print("Robots blocked:", int((~df_robot_validated["robots_allowed"]).sum()))
df_robot_validated.head(3)

Loaded robots results from cache.
Robots allowed: 496
Robots blocked: 4


,company_id,name,country,size_bucket,domain,final_url,robots_url,robots_status_code,robots_allowed,robots_reason,robots_attempts
0,C0003,austal,australia,enterprise,austal.com,https://www.austal.com,https://www.austal.com/robots.txt,200.0,True,allowed,1.0
1,C0007,billabong,australia,enterprise,billabong.com,https://www.billabong.com,https://www.billabong.com/robots.txt,200.0,True,allowed,1.0
2,C0011,australian government,australia,enterprise,australia.gov.au,https://my.gov.au/en/services,https://my.gov.au/robots.txt,200.0,True,allowed,1.0


## 5. Ticket 10269: homepage validation

In [52]:
async def fetch_company_homepage(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    final_url = company["final_url"]

    try:
        async with request_semaphore:
            response = await client.get(
                final_url,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

        status_code = response.status_code
        content_type = response.headers.get("Content-Type", "").lower()

        if status_code >= 400:
            error = f"homepage_http_error_{status_code}"
        elif "text/html" not in content_type:
            error = f"homepage_not_html_{content_type}"
        else:
            error = ""

        return {
            "company_id": company["company_id"],
            "homepage_status_code": status_code,
            "homepage_fetch_url": clean_internal_url(str(response.url)),
            "homepage_error": error,
        }

    except httpx.HTTPError as error:
        return {
            "company_id": company["company_id"],
            "homepage_status_code": None,
            "homepage_fetch_url": None,
            "homepage_error": f"homepage_fetch_error_{type(error).__name__}",
        }


async def run_homepage_fetch(df_input: pd.DataFrame) -> pd.DataFrame:
    request_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT},
        follow_redirects=True,
    ) as client:
        tasks = [
            fetch_company_homepage(
                company=row,
                client=client,
                request_semaphore=request_semaphore,
            )
            for _, row in df_input.iterrows()
        ]

        results = await asyncio.gather(*tasks)

    return pd.DataFrame(results)
df_homepage_input = df_robot_validated.loc[
    df_robot_validated["robots_allowed"]
].copy()

if HOMEPAGE_OUTPUT_CSV.exists() and not RERUN_HOMEPAGES:
    df_homepage_cached = pd.read_csv(HOMEPAGE_OUTPUT_CSV)

    requested_company_ids = set(df_homepage_input["company_id"])
    cached_company_ids = set(df_homepage_cached["company_id"])
    missing_company_ids = requested_company_ids - cached_company_ids

    df_homepage_fetch = df_homepage_cached[
        df_homepage_cached["company_id"].isin(requested_company_ids)
    ].copy()

    if missing_company_ids:
        df_missing_homepages = df_homepage_input[
            df_homepage_input["company_id"].isin(missing_company_ids)
        ].copy()

        df_new_homepage_results = await run_homepage_fetch(
            df_missing_homepages
        )

        df_new_homepage_fetch = df_missing_homepages.merge(
            df_new_homepage_results,
            on="company_id",
            how="left",
            validate="one_to_one",
        )

        df_homepage_fetch = pd.concat(
            [df_homepage_fetch, df_new_homepage_fetch],
            ignore_index=True,
        )

        df_homepage_cached = pd.concat(
            [df_homepage_cached, df_new_homepage_fetch],
            ignore_index=True,
        ).drop_duplicates(
            subset=["company_id"],
            keep="last",
        )

        df_homepage_cached.to_csv(
            HOMEPAGE_OUTPUT_CSV,
            index=False,
        )

    print("Loaded homepage results from cache.")

else:
    df_homepage_results = await run_homepage_fetch(df_homepage_input)

    df_homepage_fetch = df_homepage_input.merge(
        df_homepage_results,
        on="company_id",
        how="left",
        validate="one_to_one",
    )

    df_homepage_fetch.to_csv(
        HOMEPAGE_OUTPUT_CSV,
        index=False,
    )

print(
    "Valid homepages:",
    int(df_homepage_fetch["homepage_error"].fillna("").eq("").sum()),
)
df_homepage_fetch.head(3)

Loaded homepage results from cache.
Valid homepages: 486


,company_id,name,country,size_bucket,domain,final_url,robots_url,robots_status_code,robots_allowed,robots_reason,robots_attempts,homepage_status_code,homepage_fetch_url,homepage_error,homepage_attempts
0,C0003,austal,australia,enterprise,austal.com,https://www.austal.com,https://www.austal.com/robots.txt,200.0,True,allowed,1.0,200.0,https://www.austal.com,NaN,1.0
1,C0007,billabong,australia,enterprise,billabong.com,https://www.billabong.com,https://www.billabong.com/robots.txt,200.0,True,allowed,1.0,200.0,https://www.billabong.com,NaN,1.0
2,C0011,australian government,australia,enterprise,australia.gov.au,https://my.gov.au/en/services,https://my.gov.au/robots.txt,200.0,True,allowed,1.0,200.0,https://my.gov.au/en/services,NaN,1.0


## 6. Load the embedding model once

In [53]:
model = SentenceTransformer(MODEL_NAME)

link_theme_embeddings = model.encode(
    TARGET_LINK_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

positive_content_theme_embeddings = model.encode(
    TARGET_CONTENT_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

negative_content_theme_embeddings = model.encode(
    NEGATIVE_CONTENT_THEMES,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8447.19it/s]


## 7. Reusable semantic scoring

In [54]:
def score_discovered_links(
    links: list[dict[str, str]],
) -> list[dict[str, Any]]:
    if not links:
        return []

    valid_links: list[dict[str, str]] = []
    feature_texts: list[str] = []

    for link in links:
        feature_text = build_feature_text(
            anchor_text=link["anchor_text"],
            url_path=link["url_path"],
        )

        if not feature_text.strip():
            continue

        valid_links.append(link)
        feature_texts.append(feature_text)

    if not valid_links:
        return []

    feature_embeddings = model.encode(
        feature_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    cosine_scores = util.cos_sim(
        feature_embeddings,
        link_theme_embeddings,
    )

    max_scores, best_theme_indexes = torch.max(
        cosine_scores,
        dim=1,
    )

    scored_links: list[dict[str, Any]] = []

    for link, feature_text, score, theme_index in zip(
        valid_links,
        feature_texts,
        max_scores.cpu().tolist(),
        best_theme_indexes.cpu().tolist(),
    ):
        scored_links.append({
            **link,
            "feature_text": feature_text,
            "similarity_score": float(score),
            "matched_theme": TARGET_LINK_THEMES[theme_index],
        })

    return scored_links


def score_page_content(
    page_text: str,
) -> dict[str, float]:
    text = str(page_text).strip()[:CONTENT_SCORING_CHARACTERS]

    if not text:
        return {
            "content_positive_score": float("nan"),
            "content_negative_score": float("nan"),
            "semantic_margin": float("nan"),
        }

    page_embedding = model.encode(
        [text],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    positive_score = (
        util.cos_sim(
            page_embedding,
            positive_content_theme_embeddings,
        )
        .max()
        .cpu()
        .item()
    )

    negative_score = (
        util.cos_sim(
            page_embedding,
            negative_content_theme_embeddings,
        )
        .max()
        .cpu()
        .item()
    )

    return {
        "content_positive_score": float(positive_score),
        "content_negative_score": float(negative_score),
        "semantic_margin": float(
            positive_score - negative_score
        ),
    }

## 8. Relevance-guided BFS crawler

The queue is FIFO, so pages remain depth ordered. For each fetched page, only the strongest relevant children are added. This is a bounded, relevance-filtered BFS rather than an unrestricted whole-site crawl.

In [55]:
async def load_company_robots_parser(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> RobotFileParser | None:
    robots_url = company["robots_url"]

    async with request_semaphore:
        status_code, robots_text, fetch_error = await fetch_robots_text(
            client=client,
            robots_url=robots_url,
        )

    if fetch_error or status_code is None or status_code >= 400:
        return None

    try:
        return build_robots_parser(robots_url, robots_text)
    except Exception:
        return None


async def crawl_company_website(
    company: pd.Series,
    client: httpx.AsyncClient,
    request_semaphore: asyncio.Semaphore,
) -> list[dict[str, Any]]:
    homepage_url = clean_internal_url(company["homepage_fetch_url"])

    robots_parser = await load_company_robots_parser(
        company=company,
        client=client,
        request_semaphore=request_semaphore,
    )

    queue = deque([
        {
            "url": homepage_url,
            "parent_url": None,
            "depth": 0,
            "discovery_score": 1.0,
            "matched_theme": "homepage",
            "anchor_text": "",
            "feature_text": "homepage",
        }
    ])

    visited: set[str] = set()
    queued: set[str] = {homepage_url}
    results: list[dict[str, Any]] = []

    while queue and len(visited) < MAX_PAGES_PER_COMPANY:
        current = queue.popleft()
        current_url = current["url"]
        queued.discard(current_url)

        if current_url in visited:
            continue

        if current["depth"] > MAX_CRAWL_DEPTH:
            continue

        if is_excluded_url(current_url):
            continue

        if (
            robots_parser is not None
            and not robots_parser.can_fetch(USER_AGENT, current_url)
        ):
            results.append({
                "company_id": company["company_id"],
                "name": company["name"],
                "country": company["country"],
                "domain": company["domain"],
                "url": current_url,
                "fetch_url": None,
                "parent_url": current["parent_url"],
                "depth": current["depth"],
                "anchor_text": current["anchor_text"],
                "feature_text": current["feature_text"],
                "discovery_score": current["discovery_score"],
                "matched_theme": current["matched_theme"],
                "status_code": None,
                "content_type": None,
                "page_text": "",
                "content_positive_score":np.nan,
                "content_negative_score":np.nan,
                "semantic_margin":np.nan,
                "links_discovered": 0,
                "links_queued": 0,
                "crawl_error": "robots_blocked",
            })
            visited.add(current_url)
            continue

        visited.add(current_url)

        try:
            async with request_semaphore:
                response = await client.get(
                    current_url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                )

            status_code = response.status_code
            content_type = response.headers.get("Content-Type", "").lower()
            fetch_url = clean_internal_url(str(response.url))
            if not is_same_homepage_domain(fetch_url, homepage_url):
                results.append({
                    "company_id": company["company_id"],
                    "name": company["name"],
                    "country": company["country"],
                    "domain": company["domain"],
                    "url": current_url,
                    "fetch_url": fetch_url,
                    "parent_url": current["parent_url"],
                    "depth": current["depth"],
                    "anchor_text": current["anchor_text"],
                    "feature_text": current["feature_text"],
                    "discovery_score": current["discovery_score"],
                    "matched_theme": current["matched_theme"],
                    "status_code": response.status_code,
                    "content_type": response.headers.get("Content-Type", "").lower(),
                    "page_text": "",
                    "content_positive_score": np.nan,
                    "content_negative_score": np.nan,
                    "semantic_margin": np.nan,
                    "links_discovered": 0,
                    "links_queued": 0,
                    "crawl_error": "redirected_outside_company_domain",
                })
                continue

            if status_code >= 400:
                crawl_error = f"http_error_{status_code}"
                page_text = ""
            elif "text/html" not in content_type:
                crawl_error = f"not_html_{content_type}"
                page_text = ""
            else:
                page_text = clean_page_text(response.text)

                page_character_count = len(page_text)
                page_word_count = len(page_text.split())

                if page_character_count < MIN_PAGE_TEXT_CHARACTERS:
                    crawl_error = "page_text_too_short"
                elif page_word_count < MIN_PAGE_TEXT_WORDS:
                    crawl_error = "page_word_count_too_low"
                else:
                    crawl_error = ""

            if crawl_error:
                results.append({
                    "company_id": company["company_id"],
                    "name": company["name"],
                    "country": company["country"],
                    "domain": company["domain"],
                    "url": current_url,
                    "fetch_url": fetch_url,
                    "parent_url": current["parent_url"],
                    "depth": current["depth"],
                    "anchor_text": current["anchor_text"],
                    "feature_text": current["feature_text"],
                    "discovery_score": current["discovery_score"],
                    "matched_theme": current["matched_theme"],
                    "status_code": status_code,
                    "content_type": content_type,
                    "page_text": "",
                    "content_positive_score": np.nan,
"content_negative_score":np.nan,
"semantic_margin":np.nan,
                    "links_discovered": 0,
                    "links_queued": 0,
                    "crawl_error": crawl_error,
                })
                continue

            content_score = score_page_content(page_text)

            child_links: list[dict[str, str]] = []
            selected_children: list[dict[str, Any]] = []

            if current["depth"] < MAX_CRAWL_DEPTH:
                child_links = extract_internal_links_from_page(
                    html=response.text,
                    current_page_url=fetch_url,
                    allowed_domain_url=homepage_url,
                )

                scored_children = score_discovered_links(child_links)

                eligible_children = [
                    child
                    for child in scored_children
                    if child["similarity_score"] >= MIN_LINK_SIMILARITY
                    and child["internal_url"] not in visited
                    and child["internal_url"] not in queued
                ]

                # Avoid sorting every child when only a small fixed number is needed.
                selected_children = heapq.nlargest(
                    MAX_CHILDREN_PER_PAGE,
                    eligible_children,
                    key=lambda child: child["similarity_score"],
                )

                for child in selected_children:
                    child_url = child["internal_url"]

                    queue.append({
                        "url": child_url,
                        "parent_url": fetch_url,
                        "depth": current["depth"] + 1,
                        "discovery_score": child["similarity_score"],
                        "matched_theme": child["matched_theme"],
                        "anchor_text": child["anchor_text"],
                        "feature_text": child["feature_text"],
                    })
                    queued.add(child_url)

            results.append({
                "company_id": company["company_id"],
                "name": company["name"],
                "country": company["country"],
                "domain": company["domain"],
                "url": current_url,
                "fetch_url": fetch_url,
                "parent_url": current["parent_url"],
                "depth": current["depth"],
                "anchor_text": current["anchor_text"],
                "feature_text": current["feature_text"],
                "discovery_score": current["discovery_score"],
                "matched_theme": current["matched_theme"],
                "status_code": status_code,
                "content_type": content_type,
                "page_text": page_text,
                "content_positive_score": (
                content_score["content_positive_score"]
                ),
                "content_negative_score": (
                    content_score["content_negative_score"]
                ),
                "semantic_margin": (
                    content_score["semantic_margin"]
                ),
               
                "links_discovered": len(child_links),
                "links_queued": len(selected_children),
                "crawl_error": "",
            })

        except httpx.HTTPError as error:
            results.append({
                "company_id": company["company_id"],
                "name": company["name"],
                "country": company["country"],
                "domain": company["domain"],
                "url": current_url,
                "fetch_url": None,
                "parent_url": current["parent_url"],
                "depth": current["depth"],
                "anchor_text": current["anchor_text"],
                "feature_text": current["feature_text"],
                "discovery_score": current["discovery_score"],
                "matched_theme": current["matched_theme"],
                "status_code": None,
                "content_type": None,
                "page_text": "",
                "content_positive_score": np.nan,
                "content_negative_score": np.nan,
                "semantic_margin": np.nan,
                "links_discovered": 0,
                "links_queued": 0,
                "crawl_error": f"fetch_error_{type(error).__name__}",
            })

    return results


async def run_company_crawls(df_input: pd.DataFrame) -> pd.DataFrame:
    request_semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    async with httpx.AsyncClient(
        headers={"User-Agent": USER_AGENT},
        follow_redirects=True,
    ) as client:
        tasks = [
            crawl_company_website(
                company=row,
                client=client,
                request_semaphore=request_semaphore,
            )
            for _, row in df_input.iterrows()
        ]

        company_results = await asyncio.gather(*tasks)

    flattened_results = [
        page
        for company_pages in company_results
        for page in company_pages
    ]

    return pd.DataFrame(flattened_results)

## 9. Select crawl input and run

In [56]:
df_crawl_input = df_homepage_fetch.loc[
    df_homepage_fetch["robots_allowed"]
    & df_homepage_fetch["homepage_error"].fillna("").eq("")
    & df_homepage_fetch["homepage_fetch_url"].notna()
].copy()

print("Companies selected for crawl:", len(df_crawl_input))

if CRAWL_AUDIT_OUTPUT_CSV.exists() and not RERUN_CRAWL:
    df_crawl_cached = pd.read_csv(CRAWL_AUDIT_OUTPUT_CSV)

    requested_company_ids = set(df_crawl_input["company_id"])
    cached_company_ids = set(df_crawl_cached["company_id"])
    missing_company_ids = requested_company_ids - cached_company_ids

    df_crawled_pages = df_crawl_cached[
        df_crawl_cached["company_id"].isin(requested_company_ids)
    ].copy()

    if missing_company_ids:
        df_missing_crawl = df_crawl_input[
            df_crawl_input["company_id"].isin(missing_company_ids)
        ].copy()

        df_new_crawled_pages = await run_company_crawls(
            df_missing_crawl
        )

        df_crawled_pages = pd.concat(
            [df_crawled_pages, df_new_crawled_pages],
            ignore_index=True,
        )

        df_crawl_cached = pd.concat(
            [df_crawl_cached, df_new_crawled_pages],
            ignore_index=True,
        ).drop_duplicates(
            subset=["company_id", "url"],
            keep="last",
        )

        df_crawl_cached.to_csv(
            CRAWL_AUDIT_OUTPUT_CSV,
            index=False,
        )

    print("Loaded crawl results from cache.")

else:
    df_crawled_pages = await run_company_crawls(df_crawl_input)
    df_crawled_pages.to_csv(CRAWL_AUDIT_OUTPUT_CSV, index=False)

print("Pages visited:", len(df_crawled_pages))
print("Companies crawled:", df_crawled_pages["company_id"].nunique())

df_crawled_pages[
    [
        "company_id",
        "depth",
        "url",
        "parent_url",
        "discovery_score",
        "content_positive_score",
        "content_negative_score",
        "semantic_margin",
        "links_discovered",
        "links_queued",
        "crawl_error",
    ]
].sort_values(
    ["company_id", "depth", "discovery_score"],
    ascending=[True, True, False],
).head(30)

Companies selected for crawl: 486
Loaded crawl results from cache.
Pages visited: 4042
Companies crawled: 486


,company_id,depth,url,parent_url,discovery_score,content_positive_score,content_negative_score,semantic_margin,links_discovered,links_queued,crawl_error
500,C0001,0,https://www.sustainableaustralianbeef.com.au,NaN,1.000000,0.348786,0.248983,0.099802,32,2,NaN
501,C0001,1,https://www.sustainableaustralianbeef.com.au/g...,https://www.sustainableaustralianbeef.com.au,0.547833,0.450112,0.390799,0.059313,31,2,NaN
502,C0001,1,https://www.sustainableaustralianbeef.com.au/p...,https://www.sustainableaustralianbeef.com.au,0.540156,0.426145,0.302226,0.123919,31,2,NaN
503,C0001,2,https://www.sustainableaustralianbeef.com.au/e...,https://www.sustainableaustralianbeef.com.au/g...,0.530341,0.345458,0.313252,0.032206,31,2,NaN
504,C0001,2,https://www.sustainableaustralianbeef.com.au/e...,https://www.sustainableaustralianbeef.com.au/g...,0.526043,0.405247,0.322673,0.082574,31,2,NaN
505,C0001,2,https://www.sustainableaustralianbeef.com.au/t...,https://www.sustainableaustralianbeef.com.au/p...,0.525402,0.327156,0.235491,0.091664,31,2,NaN
506,C0001,2,https://www.sustainableaustralianbeef.com.au/e...,https://www.sustainableaustralianbeef.com.au/p...,0.513908,0.373810,0.285572,0.088238,31,2,NaN
507,C0001,3,https://www.sustainableaustralianbeef.com.au/g...,https://www.sustainableaustralianbeef.com.au/e...,0.509040,0.287894,0.264375,0.023519,0,0,NaN
508,C0001,3,https://www.sustainableaustralianbeef.com.au/g...,https://www.sustainableaustralianbeef.com.au/e...,0.506368,0.416991,0.323861,0.093130,0,0,NaN
509,C0001,3,https://www.sustainableaustralianbeef.com.au/e...,https://www.sustainableaustralianbeef.com.au/e...,0.500022,0.385391,0.303640,0.081751,0,0,NaN


## 10. Select one final values-related page per company

In [ ]:
df_crawled_pages["excluded_final_url"] = (
    df_crawled_pages["url"]
    .fillna("")
    .apply(is_excluded_url)
)

df_crawled_pages = df_crawled_pages.loc[
    ~df_crawled_pages["excluded_final_url"]
].copy()

df_valid_pages = df_crawled_pages.loc[
    df_crawled_pages["crawl_error"].fillna("").eq("")
    & df_crawled_pages["page_text"].fillna("").str.strip().ne("")
    & df_crawled_pages["content_positive_score"].notna()
    & df_crawled_pages["content_negative_score"].notna()
    & df_crawled_pages["semantic_margin"].notna()
     & (
        df_crawled_pages[
            "content_positive_score"
        ] >= MIN_CONTENT_SIMILARITY
    )
    & (
        df_crawled_pages[
            "semantic_margin"
        ] >= MIN_SEMANTIC_MARGIN
    )
].copy()

POSITIVE_VALUE_PATH_PATTERNS = [
    r"/values?(?:/|$)",
    r"/core-values?(?:/|$)",
    r"/our-values?(?:/|$)",
    r"/werte(?:/|$)",
    r"/nos-valeurs(?:/|$)",
    r"/valeurs(?:/|$)",
    r"/principles?(?:/|$)",
    r"/beliefs?(?:/|$)",
    r"/mission-and-values?(?:/|$)",
    r"/vision-and-values?(?:/|$)",
    r"/purpose-and-values?(?:/|$)",
    r"/mission-vision-values?(?:/|$)",
]


def calculate_value_path_bonus(url: str) -> float:
    path = get_url_path(url).lower()

    if any(
        re.search(pattern, path)
        for pattern in POSITIVE_VALUE_PATH_PATTERNS
    ):
        return 0.15

    return 0.0


df_valid_pages["value_path_bonus"] = (
    df_valid_pages["url"]
    .fillna("")
    .apply(calculate_value_path_bonus)
)

# Calculate text-based evidence for every candidate page
page_value_evidence = (
    df_valid_pages["page_text"]
    .apply(calculate_value_evidence)
    .apply(pd.Series)
)

df_valid_pages = pd.concat(
    [
        df_valid_pages.reset_index(drop=True),
        page_value_evidence.reset_index(drop=True),
    ],
    axis=1,
)

# Multilingual acceptance rule
df_valid_pages["contains_value_signal"] = (
    (df_valid_pages["value_section_matches"] >= 1)
    | (df_valid_pages["named_value_matches"] >= 3)
    | (
        (df_valid_pages["content_positive_score"] >= 0.50)
        & (df_valid_pages["semantic_margin"] >= 0.15)
    )
)

# Reject pages that do not have enough values-related evidence
df_valid_pages = df_valid_pages.loc[
    df_valid_pages["contains_value_signal"]
].copy()

# Rank the remaining valid pages
df_valid_pages["final_selection_score"] = (
    df_valid_pages["semantic_margin"]
    + df_valid_pages["value_path_bonus"]
    + (0.10 * df_valid_pages["content_positive_score"])
    + (0.20 * df_valid_pages["value_section_matches"])
    + (0.03 * df_valid_pages["named_value_matches"])
)

production_final_df = (
    df_valid_pages
    .sort_values(
        by=[
            "company_id",
            "final_selection_score",
            "semantic_margin",
            "content_positive_score",
            "depth",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=["company_id"],
        keep="first",
    )
    .rename(
        columns={
            "url": "internal_url",
            "fetch_url": "candidate_fetch_url",
            "status_code": "candidate_status_code",
            "page_text": "candidate_text",
            "crawl_error": "candidate_error",
        }
    )
    .reset_index(drop=True)
)

print("Final candidate pages:", len(production_final_df))
production_final_df[
    [
        "company_id",
        "name",
        "country",
        "depth",
        "internal_url",
        "discovery_score",
       "content_positive_score",
"content_negative_score",
"semantic_margin",
        "matched_theme",
    ]
].head(20)

Final candidate pages: 58


,company_id,name,country,depth,internal_url,discovery_score,content_positive_score,content_negative_score,semantic_margin,matched_theme
0,C0005,manpowergroup australia,australia,1,https://www.manpowergroup.com.au/ethics,0.902335,0.500997,0.273845,0.227152,ethical principles
1,C0006,bluescope,australia,1,https://www.bluescope.com/our-company/how-we-work,0.813525,0.473013,0.298331,0.174682,the way we work
2,C0023,ausgroup - agc | mas,australia,1,https://altrad.com.au/about/our-company,0.686790,0.500810,0.316141,0.184668,company culture
3,C0032,national protective services,australia,1,https://www.nationalprotectiveservices.com.au/...,0.846157,0.521338,0.399619,0.121718,our purpose and values
4,C0042,city of melville,australia,2,https://www.melvillecity.com.au/our-city/caree...,0.586329,0.446662,0.299178,0.147485,workplace culture
5,C0045,quantium,australia,1,https://quantium.com/about-us-values,0.958666,0.445288,0.297723,0.147565,our purpose and values
6,C0064,grupo ccr,brazil,3,https://www.motiva.com.br/instituto,0.541487,0.443254,0.299370,0.143884,our purpose
7,C0074,fiemg,brazil,3,https://www.fiemg.com.br/integridade,0.651382,0.602040,0.428006,0.174034,ethical principles
8,C0086,atem distribuidora,brazil,2,https://atem.com.br/sistema-de-integridade,0.560650,0.579903,0.442646,0.137258,ethical principles
9,C0158,"jovision technology co,. ltd.",china,2,https://ru.jovisionsecurity.com/about_us.html,0.581303,0.467449,0.317773,0.149676,our way


## 11. Extract a focused values-text window

In [ ]:
VALUES_START_PATTERNS = [
    r"\bour values\b",
    r"\bcore values\b",
    r"\bcompany values\b",
    r"\bguiding principles\b",
    r"\bwhat we believe\b",
    r"\bwhat we stand for\b",
    r"\bmission, vision and values\b",
    r"\bmission vision and values\b",
    r"\bvalues and behaviours\b",
    r"\bvalues and behaviors\b",
]


def extract_candidate_value_window(
    text: str,
    *,
    before_characters: int = 100,
    after_characters: int = 4_000,
) -> str:
    source_text = re.sub(r"\s+", " ", str(text)).strip()

    if not source_text:
        return ""

    lower_text = source_text.lower()
    match_positions = []

    for pattern in VALUES_START_PATTERNS:
        match = re.search(pattern, lower_text)

        if match:
            match_positions.append(match.start())

    if not match_positions:
        return source_text[:2_500]

    first_match = min(match_positions)
    start = max(0, first_match - before_characters)
    end = min(len(source_text), first_match + after_characters)

    return source_text[start:end].strip()


VALUE_SECTION_PATTERNS = [
    r"\bour values\b",
    r"\bcore values\b",
    r"\bcompany values\b",
    r"\bour principles\b",
    r"\bguiding principles\b",
    r"\bwhat we believe\b",
    r"\bwhat we stand for\b",
    r"\bmission.{0,40}vision.{0,40}values\b",
    r"\bpurpose.{0,40}values\b",
    r"\bvalues and behaviours\b",
    r"\bvalues and behaviors\b",
]

VALUE_WORD_PATTERNS = [
    r"\bintegrity\b",
    r"\brespect\b",
    r"\baccountability\b",
    r"\bexcellence\b",
    r"\bcollaboration\b",
    r"\binnovation\b",
    r"\btrust\b",
    r"\binclusion\b",
    r"\bsustainability\b",
    r"\bcommitment\b",
    r"\bcourage\b",
    r"\bsafety\b",
    r"\bcitizenship\b",
]


def calculate_value_evidence(text: str) -> dict[str, Any]:
    cleaned_text = re.sub(r"\s+", " ", str(text)).strip().lower()

    section_matches = sum(
        bool(re.search(pattern, cleaned_text))
        for pattern in VALUE_SECTION_PATTERNS
    )

    value_word_matches = sum(
        bool(re.search(pattern, cleaned_text))
        for pattern in VALUE_WORD_PATTERNS
    )

    has_explicit_value_section = section_matches >= 1
    has_multiple_named_values = value_word_matches >= 3

    accepted = (
        has_explicit_value_section
        or has_multiple_named_values
    )

    return {
        "value_section_matches": section_matches,
        "named_value_matches": value_word_matches,
        "contains_value_signal": accepted,
    }


production_final_df["candidate_text_full"] = (
    production_final_df["candidate_text"]
)

production_final_df["candidate_text"] = (
    production_final_df["candidate_text"]
    .apply(extract_candidate_value_window)
)


production_final_df["quality_status"] = np.where(
    production_final_df["contains_value_signal"],
    "accepted",
    "manual_review",
)

production_final_df.to_csv(PRODUCTION_FINAL_CSV, index=False)

## 12. Translation configuration

In [59]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "translategemma:4b"
MAX_CONCURRENT_TRANSLATIONS = 2
MAX_RETRIES = 3
TRANSLATION_TIMEOUT_SECONDS = 120
DetectorFactory.seed = 0

## 13. Language detection and translation

In [60]:
def detect_source_language(text: str) -> tuple[str, str]:
    detected_code = detect(text)

    code_aliases = {
        "zh-cn": "zh-Hans",
        "zh-tw": "zh-Hant",
    }

    detected_code = code_aliases.get(detected_code, detected_code)

    language_name_overrides = {
        "zh-Hans": "Chinese (Simplified)",
        "zh-Hant": "Chinese (Traditional)",
    }

    if detected_code in language_name_overrides:
        language_name = language_name_overrides[detected_code]
    else:
        base_code = detected_code.split("-")[0]
        language = pycountry.languages.get(alpha_2=base_code)
        language_name = language.name if language else detected_code

    return detected_code, language_name


def build_translation_prompt(
    text: str,
    source_code: str,
    source_language: str,
) -> str:
    return (
        f"Translate the following company values text from "
        f"{source_language} ({source_code}) into English.\n\n"
        "Rules:\n"
        "- Translate literally and completely.\n"
        "- Do not summarise.\n"
        "- Do not reorganise the content.\n"
        "- Do not add headings unless they exist in the source.\n"
        "- Do not explain the translation.\n"
        "- Return only the translated text.\n\n"
        f"SOURCE TEXT:\n{text}"
    )

async def translate_value(
    text: str,
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    source_text = str(text).strip()

    try:
        source_code, source_language = detect_source_language(source_text)
    except Exception as error:
        return {
            "source_text": source_text,
            "detected_language": "unknown",
            "translation_needed": None,
            "translated_value_text": None,
            "translation_method": "not_started",
            "translation_status": "language_detection_failed",
            "translation_error": type(error).__name__,
        }

    if source_code == "en":
        return {
            "source_text": source_text,
            "detected_language": source_code,
            "translation_needed": False,
            "translated_value_text": source_text,
            "translation_method": "not_required",
            "translation_status": "already_english",
            "translation_error": "",
        }

    payload = {
        "model": OLLAMA_MODEL,
        "messages": [{
            "role": "user",
            "content": build_translation_prompt(
                text=source_text,
                source_code=source_code,
                source_language=source_language,
            ),
        }],
        "stream": False,
        "options": {"temperature": 0},
    }

    async with semaphore:
        for attempt in range(MAX_RETRIES):
            try:
                response = await client.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json=payload,
                )
                response.raise_for_status()

                translated_text = response.json()["message"]["content"].strip()

                if not translated_text:
                    raise ValueError("Empty translation returned")

                commentary_prefixes = [
                    "here's a translation",
                    "here is a translation",
                    "translation:",
                    "the translation is",
                ]

                translated_text_lower = translated_text.lower()

                if any(
                    translated_text_lower.startswith(prefix)
                    for prefix in commentary_prefixes
                ):
                    raise ValueError("Translation returned commentary")

                return {
                    "source_text": source_text,
                    "detected_language": source_code,
                    "translation_needed": True,
                    "translated_value_text": translated_text,
                    "translation_method": f"ollama:{OLLAMA_MODEL}",
                    "translation_status": "translated",
                    "translation_error": "",
                }

            except Exception as error:
                if attempt == MAX_RETRIES - 1:
                    return {
                        "source_text": source_text,
                        "detected_language": source_code,
                        "translation_needed": True,
                        "translated_value_text": None,
                        "translation_method": f"ollama:{OLLAMA_MODEL}",
                        "translation_status": "failed",
                        "translation_error": type(error).__name__,
                    }

                await asyncio.sleep(2 ** attempt)


async def run_translation(texts: list[str]) -> list[dict[str, Any]]:
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_TRANSLATIONS)

    async with httpx.AsyncClient(
        timeout=TRANSLATION_TIMEOUT_SECONDS,
    ) as client:
        health_response = await client.get(f"{OLLAMA_BASE_URL}/api/tags")
        health_response.raise_for_status()

        tasks = [
            translate_value(
                text=text,
                client=client,
                semaphore=semaphore,
            )
            for text in texts
        ]

        return await asyncio.gather(*tasks)

## 14. Run translation and export

In [61]:
df_values_translated = production_final_df.loc[
    production_final_df["candidate_error"].fillna("").eq("")
    & production_final_df["candidate_text"].fillna("").str.strip().ne("")
    & production_final_df["quality_status"].eq("accepted")
].copy()

df_values_translated["_translation_key"] = (
    df_values_translated["candidate_text"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

unique_texts = (
    df_values_translated["_translation_key"]
    .drop_duplicates()
    .tolist()
)

print("Rows selected:", len(df_values_translated))
print("Unique texts:", len(unique_texts))
if TRANSLATION_CACHE_CSV.exists() and not RERUN_TRANSLATION:
    translation_cache_df = pd.read_csv(TRANSLATION_CACHE_CSV)
else:
    translation_cache_df = pd.DataFrame(
        columns=[
            "source_text",
            "detected_language",
            "translation_needed",
            "translated_value_text",
            "translation_method",
            "translation_status",
            "translation_error",
        ]
    )

cached_texts = set(
    translation_cache_df["source_text"]
    .dropna()
    .astype(str)
)

texts_to_translate = [
    text
    for text in unique_texts
    if text not in cached_texts
]

print(
    "Cached translations:",
    len(unique_texts) - len(texts_to_translate),
)
print("New translations required:", len(texts_to_translate))

if texts_to_translate:
    new_translation_results = await run_translation(
        texts_to_translate
    )
    new_translation_results_df = pd.DataFrame(
        new_translation_results
    )

    translation_cache_df = pd.concat(
        [
            translation_cache_df,
            new_translation_results_df,
        ],
        ignore_index=True,
    ).drop_duplicates(
        subset=["source_text"],
        keep="last",
    )

    translation_cache_df.to_csv(
        TRANSLATION_CACHE_CSV,
        index=False,
        encoding="utf-8-sig",
    )

translation_results_df = translation_cache_df[
    translation_cache_df["source_text"].isin(unique_texts)
].copy()

df_values_translated = (
    df_values_translated.merge(
        translation_results_df,
        left_on="_translation_key",
        right_on="source_text",
        how="left",
        validate="many_to_one",
    )
    .drop(columns=["_translation_key", "source_text"])
)

df_export = (
    df_values_translated[
        [
            "company_id",
            "name",
            "country",
            "internal_url",
            "depth",
            "content_positive_score",
            "content_negative_score",
            "semantic_margin",
            "candidate_text",
            "quality_status",
            "contains_value_signal",
            "detected_language",
            "translation_needed",
            "translated_value_text",
            "translation_method",
            "translation_status",
            "translation_error",
        ]
    ]
    .rename(columns={
        "company_id": "id",
        "detected_language": "detect_language",
    })
    .copy()
)

text_columns = df_export.select_dtypes(include=["object", "string"]).columns

for column in text_columns:
    df_export[column] = (
        df_export[column]
        .astype("string")
        .str.replace(r"[\r\n\t]+", " ", regex=True)
        .str.replace(r"\s{2,}", " ", regex=True)
        .str.strip()
    )

df_export.to_csv(
    TRANSLATION_OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL,
    lineterminator="\n",
)

print(f"Saved {len(df_export)} rows to {TRANSLATION_OUTPUT_CSV}")

Rows selected: 7
Unique texts: 7
Cached translations: 7
New translations required: 0
Saved 7 rows to ../data/processed/production_500_sample.csv


## TIME TAKEN TRIAL_1: 51m 44.4s

## TIME TAKEN TRIAL_2: 50m 44.1s

## TIME TAKE TRIAL 3:

In [63]:
# Final internal links actually selected for translation
preview_df = production_final_df[
    [
        "company_id",
        "name",
        "country",
        "internal_url",
        "depth",
        "discovery_score",
        "matched_theme",
        "content_positive_score",
        "semantic_margin",
    ]
].sort_values(
    ["company_id", "depth", "discovery_score"],
    ascending=[True, True, False],
).reset_index(drop=True)

preview_df.head(50)

,company_id,name,country,internal_url,depth,discovery_score,matched_theme,content_positive_score,semantic_margin
0,C0005,manpowergroup australia,australia,https://www.manpowergroup.com.au/ethics,1,0.902335,ethical principles,0.500997,0.227152
1,C0006,bluescope,australia,https://www.bluescope.com/our-company/how-we-work,1,0.813525,the way we work,0.473013,0.174682
2,C0023,ausgroup - agc | mas,australia,https://altrad.com.au/about/our-company,1,0.686790,company culture,0.500810,0.184668
3,C0032,national protective services,australia,https://www.nationalprotectiveservices.com.au/...,1,0.846157,our purpose and values,0.521338,0.121718
4,C0042,city of melville,australia,https://www.melvillecity.com.au/our-city/caree...,2,0.586329,workplace culture,0.446662,0.147485
5,C0045,quantium,australia,https://quantium.com/about-us-values,1,0.958666,our purpose and values,0.445288,0.147565
6,C0064,grupo ccr,brazil,https://www.motiva.com.br/instituto,3,0.541487,our purpose,0.443254,0.143884
7,C0074,fiemg,brazil,https://www.fiemg.com.br/integridade,3,0.651382,ethical principles,0.602040,0.174034
8,C0086,atem distribuidora,brazil,https://atem.com.br/sistema-de-integridade,2,0.560650,ethical principles,0.579903,0.137258
9,C0158,"jovision technology co,. ltd.",china,https://ru.jovisionsecurity.com/about_us.html,2,0.581303,our way,0.467449,0.149676
